# Demanda e zonas (Gold)

Refinamento da Silver em tabelas prontas para o núcleo Python (`src/tcc_frota/`). A Gold **não** faz previsão, alocação nem simulação: apenas agrega contagens históricas com rótulos de base temporal e espacial.

## Fluxo dos dados

* Origens: `tb_posicoes_motoristas`, `tb_corridas` e `tb_solicitacoes` da Silver
* Destinos em `cat_mobilidade_urbana_gold.db_frota`: `tb_dim_zonas_h3`, `tb_demanda_zona_hora`, `tb_demanda_regional_hora`

## Cuidados para o D+1 (RN020, RN030)

* As contagens estão por **data local**: o Python deve filtrar somente datas anteriores ao corte antes de estimar.
* `tb_dim_zonas_h3` é só geometria (células e centróides). Não contém volume, para não vazar informação futura na escolha das zonas; o ajuste da malha deve usar apenas o treino.
* `tb_demanda_zona_hora` conta **corridas atendidas com origem confiável**. Pedidos não atendidos não têm posição nesta fonte e aparecem apenas em `tb_demanda_regional_hora`, sem zona.

In [0]:
# No Databricks `spark` já existe; no VS Code o Databricks Connect cria uma sessão serverless.
try:
    spark  # noqa: F821
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless(True).getOrCreate()

from pyspark.sql import Window as W
from pyspark.sql import functions as F

# Todos os carimbos da fonte têm offset -03:00; datas e horas locais usam o fuso informado pelo proponente.
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

BRONZE = "cat_mobilidade_urbana_bronze.db_frota"
SILVER = "cat_mobilidade_urbana_silver.db_frota"
GOLD = "cat_mobilidade_urbana_gold.db_frota"


def segundos_entre(inicio, fim):
    """Diferença em segundos (com fração) entre dois timestamps."""
    return F.col(fim).cast("double") - F.col(inicio).cast("double")


def haversine_km(lat1, lon1, lat2, lon2):
    """Mesmo Haversine de src/tcc_frota/distance.py (R = 6371,0088 km).

    Linha reta entre dois pontos: estimativa geodésica, nunca rodagem observada (RN012).
    """
    la1, lo1, la2, lo2 = (F.radians(F.col(c)) for c in (lat1, lon1, lat2, lon2))
    h = F.pow(F.sin((la2 - la1) / 2), 2) + F.cos(la1) * F.cos(la2) * F.pow(F.sin((lo2 - lo1) / 2), 2)
    return F.lit(6371.0088) * 2 * F.asin(F.least(F.lit(1.0), F.sqrt(h)))


def pseudonimo(coluna):
    """Pseudônimo determinístico (SHA-256 truncado) para tabelas Gold; não é anonimização."""
    return F.substring(F.sha2(F.concat(F.lit("tcc-frota|"), F.col(coluna).cast("string")), 256), 1, 16)


def gravar(df, tabela, comentario):
    """Sobrescreve a tabela Delta de forma idempotente e registra o comentário."""
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

## 1. Dimensão de zonas H3

In [0]:
def centro(coluna, prefixo):
    wkt = F.expr(f"h3_centeraswkt({coluna})")
    return [
        F.regexp_extract(wkt, r"POINT\((\S+) (\S+)\)", 2).cast("double").alias(f"lat_centro_{prefixo}"),
        F.regexp_extract(wkt, r"POINT\((\S+) (\S+)\)", 1).cast("double").alias(f"lon_centro_{prefixo}"),
    ]


zonas = (
    spark.table(f"{SILVER}.tb_posicoes_motoristas")
    .where("fl_coordenada_valida AND h3_r8 IS NOT NULL")
    .select("h3_r8", "h3_r7")
    .distinct()
    .select(
        "h3_r8",
        F.expr("h3_h3tostring(h3_r8)").alias("h3_r8_hex"),
        *centro("h3_r8", "r8"),
        "h3_r7",
        F.expr("h3_h3tostring(h3_r7)").alias("h3_r7_hex"),
        *centro("h3_r7", "r7"),
    )
    .withColumn(
        "fl_bbox_grande_vitoria",
        F.col("lat_centro_r8").between(-20.85, -19.80) & F.col("lon_centro_r8").between(-40.80, -40.00),
    )
)
gravar(
    zonas,
    f"{GOLD}.tb_dim_zonas_h3",
    "Celulas H3 r8 observadas nas posicoes GPS, com pai r7 e centroides. Apenas geometria, sem volume.",
)

## 2. Demanda atendida por zona e hora

In [0]:
demanda_zona = (
    spark.table(f"{SILVER}.tb_corridas")
    .where("fl_origem_confiavel AND fl_tempos_ordenados")
    .groupBy(
        "data_local", "hora_local", "dia_semana_iso", "fl_fim_de_semana", "tipo_solicitacao",
        F.col("h3_r8_origem").alias("h3_r8"), F.col("h3_r7_origem").alias("h3_r7"),
    )
    .agg(F.count("*").alias("qtd_corridas_atendidas"))
    .withColumn("base_temporal", F.lit("dT_REGISTRO_entrada_fila_proxy_nao_homologada"))
    .withColumn("base_espacial", F.lit("gps_motorista_no_embarque"))
    .withColumn("universo", F.lit("corridas_encerradas_com_origem_confiavel"))
)
gravar(
    demanda_zona,
    f"{GOLD}.tb_demanda_zona_hora",
    "Corridas atendidas com origem confiavel por data local, hora local e celula H3 da origem. "
    "Demanda atendida, nao demanda total.",
)

## 3. Solicitações por hora e desfecho (sem zona)

In [0]:
demanda_regional = (
    spark.table(f"{SILVER}.tb_solicitacoes")
    .groupBy(
        "data_local", "hora_local", "dia_semana_iso", "fl_fim_de_semana", "id_regiao",
        "tipo_solicitacao", "grupo_desfecho", "categoria_desfecho",
    )
    .agg(F.count("*").alias("qtd_solicitacoes"))
    .withColumn("base_temporal", F.lit("dT_REGISTRO_entrada_fila_proxy_nao_homologada"))
)
gravar(
    demanda_regional,
    f"{GOLD}.tb_demanda_regional_hora",
    "Todas as solicitacoes deduplicadas por data e hora local, tipo e desfecho, sem zona. "
    "Base para cobertura temporal e nao atendimento.",
)